- Blur everything other than the ROI
- The model only needs contextual information of where the object is. 
- Just blurring however is not working, need to verify now


The first thing we need to do is decide how much of the mask should we look at (the pad of the mask we use)


I dont see a lot of difference between 5 or 10 pad. we use 5 pad. Now about blur kernel size. For now I like the higher blur. Lets go with high blur and less pad

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import shutil
from fastai.vision.all import *
from mtrain.utils import mkdir, DiskImage, DiskBooleanMask, show
from tqdm import tqdm
from mtrain.neg_mask.leveled_cropping import (
    load_crop_level_sample_from_directory,
    make_crop_level_pairs_v2,
)
from mtrain.example_dir.iterdir import get_labelled_dirs

In [ ]:
CROP_LEVEL_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level"
)
BLURRED_ROOT = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred"
)
IP_DS = BLURRED_ROOT / "clean"
B3P5 = BLURRED_ROOT / "b3p5"
B5P5 = BLURRED_ROOT / "b5p5"
B5P5_BLURS = BLURRED_ROOT / "b5p5_blur_k7"
B5P5_S1K7 = BLURRED_ROOT / "b5p5_s1k7"

In [ ]:
def get_padded_bbox_mask(mask, padding=10):
    # 1. Find the coordinates of all non-zero pixels
    coords = cv2.findNonZero(mask)
    if coords is None:
        return np.zeros_like(mask), None

    # 2. Get the standard bounding box
    x, y, w, h = cv2.boundingRect(coords)
    img_h, img_w = mask.shape[:2]

    # 3. Apply padding with boundary constraints
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(img_w, x + w + padding)
    y2 = min(img_h, y + h + padding)

    # 4. Create the new mask
    padded_mask = np.zeros_like(mask)
    cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

    return padded_mask, (x1, y1, x2, y2)


def get_blurred_artifacts(
    img_path, mask_path, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    img = cv2.imread(img_path)
    assert img is not None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = DiskBooleanMask.load(mask_path)
    new_mask, box = get_padded_bbox_mask(mask, bbox_pad)

    blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
    only_mask_unblurred = blurred.copy()

    new_mask = new_mask.astype(bool)
    only_mask_unblurred[new_mask] = img[new_mask]

    if box is not None:
        x1, y1, x2, y2 = box
        original_crop = img[y1:y2, x1:x2]
    else:
        original_crop = None

    # padded_crop = np.random.randint(0, 30, img.shape, dtype=np.uint8)  # low noise, adjust 15 to taste
    padded_crop = np.zeros(img.shape, dtype=np.uint8)
    padded_crop[new_mask] = img[new_mask]

    return {
        "mask": mask,
        "padded_mask": new_mask,
        "blurred": blurred,
        "unblurred": only_mask_unblurred,
        "img": img,
        "original_crop": original_crop,
        "padded_crop": padded_crop,
    }


def create_train_ds(
    ip_ds,
    out_dir_root,
    blur_kernel_sz,
    blur_sigma,
    bbox_pad,
    artifact_to_dump="unblurred",
):
    # remove the out ds firs
    shutil.rmtree(out_dir_root, True)
    train_dir = mkdir(out_dir_root / "train")
    mask_dir = mkdir(out_dir_root / "masks")

    for label in ["other", "trash"]:
        label_d = ip_ds / label
        label_dirs = list(label_d.glob("*"))
        for d in tqdm(label_dirs):
            if not d.is_dir() or not (d / "orig.jpg").exists():
                continue

            arts = get_blurred_artifacts(
                d / "orig.jpg",
                d / "mask.png",
                blur_kernel_sz=blur_kernel_sz,
                blur_sigma=blur_sigma,
                bbox_pad=bbox_pad,
            )
            out_arr = arts[artifact_to_dump]
            fname = f"{label}_{d.name}"
            mask = arts["padded_mask"]

            DiskImage.save(out_arr, train_dir / f"{fname}.jpg")
            DiskBooleanMask.save(mask, mask_dir / f"{fname}.png")

            # out_dir = mkdir(train_dir / label)

            # dest_file = out_dir / fname
            # DiskImage.save(out_arr, dest_file)
            # DiskBooleanMask.save(arts["padded_mask"], mask_dir / f"{dest_file.stem}.png")


In [ ]:
def create_clean_crops(crop_level_path, root_dest_dir, crop_size):
    for label in ["other", "trash"]:
        dirs = list((crop_level_path / label).glob("*"))
        for p in tqdm(dirs):
            if (
                not p.is_dir()
                or not (p / "image.jpg").exists()
                or not (p / "source_dir" / "image.jpg").exists()
            ):
                continue
            try:
                sample = load_crop_level_sample_from_directory(p)
                level_pairs = make_crop_level_pairs_v2(
                    sample, crop_size, crop_size + 100, 1
                )
            except Exception as ex:
                print(f"WARN: failure in getting crop; dir={p.name} reason={ex}")

            crop, mask = level_pairs.pairs[0]
            dest_dir = mkdir(root_dest_dir / label / p.name)
            DiskImage.save(crop, dest_dir / "orig.jpg")
            DiskBooleanMask.save(mask, dest_dir / "mask.png")

In [ ]:
# def get_blurred_artifacts(img, bbox, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5, crop_size=None):
#     assert img is not None
#     new_mask, box = get_padded_bbox_mask(img.shape, bbox, bbox_pad)
#     blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
#     only_mask_unblurred = blurred.copy()
#     new_mask = new_mask.astype(bool)
#     only_mask_unblurred[new_mask] = img[new_mask]

#     result = {
#         "padded_mask": new_mask,
#         "unblurred": only_mask_unblurred,
#         "img": img,
#     }

#     if crop_size is not None:
#         img_h, img_w = img.shape[:2]
#         cx = (bbox.x + bbox.x2) // 2
#         cy = (bbox.y + bbox.y2) // 2
#         half = crop_size // 2

#         x1 = max(0, cx - half)
#         y1 = max(0, cy - half)
#         x2 = min(img_w, x1 + crop_size)
#         y2 = min(img_h, y1 + crop_size)
#         # shift back if clamped on the far edge
#         x1 = max(0, x2 - crop_size)
#         y1 = max(0, y2 - crop_size)

#         result["crop"] = only_mask_unblurred[y1:y2, x1:x2]
#         result["crop_box"] = (x1, y1, x2, y2)

#     return result

# Manual visualize, analyse for parameters

In [ ]:
dirs_and_labels = list(get_labelled_dirs(IP_DS))

In [ ]:
idx = -1

In [ ]:
# idx += 1
idx = random.randint(0, len(dirs_and_labels))
d, lbl = dirs_and_labels[idx]
arts = get_blurred_artifacts(
    d / "orig.jpg", d / "mask.png", blur_kernel_sz=3, blur_sigma=1, bbox_pad=10
)

high_arts = get_blurred_artifacts(
    d / "orig.jpg", d / "mask.png", blur_kernel_sz=13, blur_sigma=4, bbox_pad=5
)

if arts["original_crop"] is None:
    print("original crop is None, showing mask")
    show([arts["mask"], arts["unblurred"]])
else:
    show(
        [
            arts["img"],
            arts["unblurred"],
            high_arts["unblurred"],
        ],
        (20, 20),
        ncols=3,
        axis="off",
    )
print("label", lbl)

# Generate data

In [ ]:
B3P5 = BLURRED_ROOT / "b3p5"
B5P5 = BLURRED_ROOT / "b5p5"
B5P5_BLUR_K5S3 = BLURRED_ROOT / "b5p5_blur_k5s3"
B5P5_BLUR_K15S5 = BLURRED_ROOT / "b5p5_blur_k15s5"
B5P5_BLUR_K13S4 = BLURRED_ROOT / "b5p5_blur_k13s4"

In [ ]:
B5P5_0PAD_NOISY = BLURRED_ROOT / "b5p5_0pad_noisy"
B5P5K13S5 = BLURRED_ROOT / "b5p5_0pad_k13s5"

In [ ]:
B5P5_0PAD_NOISY

In [ ]:
B5P10_BLUR_K13S4

In [ ]:
B5P10_BLUR_K3S1

In [ ]:
# B5P10_BLUR_K13S4 = BLURRED_ROOT / "b5p10_blur_k13s4"
B5P10_BLUR_K3S1 = BLURRED_ROOT / "b5p10_blur_k3s1"

# shutil.rmtree(B5P10_BLUR_K13S4)
create_train_ds(IP_DS, B5P10_BLUR_K3S1, 3, 1, 10, "unblurred")

In [ ]:
B5P10_BLUR_FULL_CLEAN = BLURRED_ROOT / "b5p10_clean-no-blur-only-mask"
create_train_ds(IP_DS, B5P10_BLUR_FULL_CLEAN, 3, 1, 10, "img")

In [ ]:
B5P5_BLUR_K13S4

In [ ]:
from mtrain.neg_mask.crops import Bbox

def get_load_coord_around_object(mask):
    # given a mask, we find where the bbox is
    # we return the coordinates of a bigger bbox around this bbox
    pass

def padded_crop(arr: np.ndarray, bbox: Bbox, pad: int) -> tuple[np.ndarray, int, int]:
    """
    Crop `arr` around `bbox` with `pad` pixels on each side, clamped to array bounds.

    Returns
    -------
    crop   : the cropped sub-array (view, not a copy)
    y1c    : actual top row used  (needed to map bbox coords into crop space)
    x1c    : actual left col used
    """
    H, W = arr.shape[:2]
    y1c = max(0, bbox.y - pad)
    y2c = min(H, bbox.y2 + pad)
    x1c = max(0, bbox.x - pad)
    x2c = min(W, bbox.x2 + pad)
    return arr[y1c:y2c, x1c:x2c], y1c, x1c

In [ ]:
mpath = '/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy/masks/other_1009233249610945_11.png'

In [ ]:
mask = DiskBooleanMask.load(mpath)

In [ ]:
def get_region_crops(mask):
    _, labels = cv2.connectedComponents(mask)
    h, w = mask.shape
    for label in range(1, labels.max() + 1):
        rows, cols = np.where(labels == label)
        r1 = max(0, rows.min())
        r2 = min(h, rows.max())
        c1 = max(0, cols.min())
        c2 = min(w, cols.max())
        yield Bbox(c1, r1, c2 - c1, r2 - r1)


pad = 130
bbox = list(get_region_crops(mask))[0]
print(bbox)

center_x, center_y = bbox.x + (bbox.w // 2), bbox.y + (bbox.h // 2)
print(center_x, center_y)

left_pad = pad // 2
right_pad = pad - left_pad

# pad to left and top, by half the pad size
left_x = max(center_x - left_pad, 0)
up_y = max(center_y - left_pad, 0)

right_x = min(left_x + pad, mask.shape[1])
down_y = min(up_y + pad, mask.shape[0])

new_mask = mask[up_y:down_y, left_x:right_x]

show([mask, new_mask])

In [ ]:
RT = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy')
RT / "masks"

In [ ]:
images = list((B5P5_0PAD_NOISY / "train").glob("*.jpg"))

In [ ]:
def get_mask_where_no_noise_should_be_added(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    ret, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)

    x0, w0, y0, h0 = None, None, None, None
    for i in range(1, num_labels):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]
        
        # Filter out tiny noise if needed
        if area > 10: 
            cv2.rectangle(gray, (x, y), (x + w, y + h), (255), 2)
            x0, y0, w0, h0 = x, y, w, h
            break

    if x0 is None or y0 is None or w0 is None or h0 is None:
        # empty mask
        return np.zeros(img_rgb.shape[:2], dtype=bool)
    
    res_mask = np.zeros(img_rgb.shape[:2], dtype=bool)
    res_mask[y0:y0+h0, x0:x0+w0] = True
    return res_mask


def get_noisy_image(img_arr, noise_max):
    mask = get_mask_where_no_noise_should_be_added(img_arr)
    noisy = np.random.randint(0, noise_max, img_arr.shape, dtype=np.uint8)
    noisy[mask] = img_arr[mask]
    return noisy


In [ ]:
image = plt.imread(images[4])
noisy = get_noisy_image(image, 80)
show([
    image, noisy
], ncols=2, cmap="gray")

In [ ]:
image = plt.imread(images[2])
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
analysis = cv2.connectedComponentsWithStats(gray, connectivity=8)
ret, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)
# (totalLabels, label_ids, values, centroid) = analysis
print(num_labels)
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    
    # Filter out tiny noise if needed
    if area > 10: 
        cv2.rectangle(gray, (x, y), (x + w, y + h), (255), 2)
        print(f"Region {i}: Center at {centroids[i]}, Area: {area}")
plt.imshow(gray)

In [ ]:
gray

In [ ]:
images = list((B5P5_BLUR_K13S4 / "train").rglob("*.jpg"))
masks = list((B5P5_BLUR_K13S4 / "masks").rglob("*.png"))
# k5s3_images = list(B5P5_0PAD_NOISY.rglob("*.jpg"))

In [ ]:
image = images[5007]
mask = B5P5_BLUR_K13S4 / "masks" / f"{image.stem}.png"
show([plt.imread(image), plt.imread(mask)], (20, 20), ncols=2)

In [ ]:
from fastai.vision.all import *


class MaskedNoiseTransform(ItemTransform):
    def __init__(self, mask_folder, noise_level=0.1):
        self.mask_folder = Path(mask_folder)
        self.noise_level = noise_level

    def encodes(self, x: PILImage):
        # 1. Get the path of the current file
        # Fastai attaches the path to the PILImage object during loading
        img_path = Path(x.path)

        # 2. Find the corresponding mask file
        # Assuming mask has the same name as the image
        mask_path = self.mask_folder / img_path.name

        # 3. Load mask and convert to tensor
        # We ensure it's a single channel (L) and normalized to 0/1
        mask = Image.open(mask_path).convert("L")
        mask_tensor = tensor(mask).float() / 255.0

        # 4. Convert image to tensor and apply noise
        img_tensor = image2tensor(x).float() / 255.0

        noise = torch.randn_like(img_tensor) * self.noise_level
        # Mask is 0 where we want noise, 1 where we don't
        noise_mask = 1 - mask_tensor

        # Apply noise logic
        noisy_img = img_tensor + (noise * noise_mask)
        noisy_img = torch.clamp(noisy_img, 0, 1)

        # Return as a fastai Image type
        return PILImage.create((noisy_img * 255).byte())

In [ ]:
PILImage.create(image)

In [ ]:
import torch


def add_masked_noise(image, mask, noise_level=0.1):
    """
    Adds Gaussian noise to an image only where the mask is 0.

    Args:
        image (torch.Tensor): The input image tensor.
        mask (torch.Tensor): Mask tensor (0 = add noise, 1 = keep original).
        noise_level (float): Standard deviation of the Gaussian noise.
    """
    # 1. Generate noise with the same shape as the image
    noise = torch.randn_like(image) * noise_level

    # 2. Create an inverse mask
    # If mask is 0 for noise, (1 - mask) is 1 for noise.
    noise_mask = 1 - mask

    # 3. Apply noise only to those specific pixels
    # We multiply the noise by the inverse mask so it's 0 everywhere else.
    noisy_image = image + (noise * noise_mask)

    # 4. Optional: Clamp to keep pixel values in valid range (e.g., [0, 1])
    return torch.clamp(noisy_image, 0, 1)


# Example usage:
# noisy_img = add_masked_noise(my_image_tensor, my_mask_tensor)

In [ ]:
# class ReNoisePadding(RandTransform):
#     order = 1

#     def __init__(self, p=1.0, subset_frac=0.2):
#         super().__init__(p=p)
#         self.subset_frac = subset_frac

#     def encodes(self, x: TensorImage):
#         np_img = x.numpy()  # (C, H, W)

#         # mask where all channels < 30
#         mask = (np_img < 30).all(axis=0)  # (H, W)

#         # randomly select a subset of masked pixels
#         mask_indices = np.where(mask)
#         n_pixels = len(mask_indices[0])
#         n_select = int(n_pixels * self.subset_frac)
#         chosen = np.random.choice(n_pixels, size=n_select, replace=False)

#         subset_mask = np.zeros_like(mask)
#         subset_mask[mask_indices[0][chosen], mask_indices[1][chosen]] = True

#         # re-randomize only the selected subset
#         new_noise = np.random.randint(0, 30, np_img.shape, dtype=np.uint8)
#         np_img[:, subset_mask] = new_noise[:, subset_mask]

#         return TensorImage(torch.from_numpy(np_img))

In [ ]:
show([], ncols=1)

In [ ]:
idx = 20
img = images[idx]

print(img.name)
img
(plt.imread(img))

# Model performance

In [ ]:
BASE = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification"
)
stage0_learn = load_learner(BASE / "unblurred_stage1_0pad_crops_b5p5_resnet18.pkl")
stage1_learn = load_learner(
    BASE / "unblurred_stage2_blurpad_crops_b5p5k5s3_resnet18.pkl"
)
stage0_learn.eval()
stage1_learn.eval()


In [ ]:
LOG_ROOT = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification"
)


def label_func(x):
    if x.startswith("other"):
        return "other"
    elif x.startswith("trash"):
        return "trash"
    else:
        raise Exception(f"bad file name {x}")


def get_dls(root_dir, log_root=LOG_ROOT):
    return ImageDataLoaders.from_name_func(
        LOG_ROOT,
        get_image_files(root_dir),
        valid_pct=0.2,
        seed=42,
        label_func=label_func,
        item_tfms=CropPad(130),
        bs=8,
    )


B5P5 = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5"
)
B5P5_BLUR_K5S3 = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_blur_k5s3"
)

stage0_learn.dls = get_dls(B5P5)
stage1_learn.dls = get_dls(B5P5_BLUR_K5S3)


In [ ]:
stage0_learn.show_results()

In [ ]:
interp = ClassificationInterpretation.from_learner(stage0_learn)
interp.plot_confusion_matrix()

In [ ]:
interp.plot_top_losses(5, nrows=1)